# How the soccer-ball (icosahedral) starting points are made

This notebook rebuilds, in plain numpy, the starting points our WW3 mesh uses:
`initial_points_icos` in `mesh/mesawave_mesh/make_mesh.m` (lines 106–144).
Each step names the matching MATLAB line.

What the notebook covers:
- the 12 corners and 20 faces of an icosahedron (the soccer ball)
- filling each face with a triangle lattice
- why every point has 6 neighbours except 12 that have 5
- how the spacing sets the number of points
- the small built-in stretch
- how the real pipeline uses these points: ocean only, several sizes, then relaxation ("shaking")

The relaxation (DistMesh, inside OceanMesh2D) is described in words only, not reproduced.

Run it with NPL Python (`/glade/u/apps/opt/conda/envs/npl/bin/python`), for example as a Jupyter kernel.
Everything here runs in a few seconds; the biggest lattice built has about 92,000 points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import ConvexHull, cKDTree

R = 6378.137   # Earth radius in km, as make_mesh.m line 114 (there in m)

## 1. The 12 corners

An icosahedron has 12 corners, 20 triangular faces and 30 edges. A soccer ball's 12 black pentagons sit where an
icosahedron's corners are. The corners have a neat formula. With the golden ratio φ = (1 + √5)/2, they are every
point of the form (0, ±1, ±φ), (±1, ±φ, 0) and (±φ, 0, ±1). We divide each one by its length so that it lies on a
sphere of radius 1. In `make_mesh.m` this is lines 118–120. Line 120 also rotates the corners by a matrix `Q`
(lines 115–117), which `tools/icos_rotation.py` chose so that 8 of the 12 corners fall on land (the comment on
lines 112–113). We skip that rotation here.

In [ ]:
g = (1 + np.sqrt(5)) / 2
V = np.array([[0, 1, g], [0, -1, g], [0, 1, -g], [0, -1, -g],
              [1, g, 0], [-1, g, 0], [1, -g, 0], [-1, -g, 0],
              [g, 0, 1], [-g, 0, 1], [g, 0, -1], [-g, 0, -1]], dtype=float)
V /= np.linalg.norm(V, axis=1, keepdims=True)    # onto the unit sphere
print(V.shape, 'all at radius 1:', np.allclose(np.linalg.norm(V, axis=1), 1))

## 2. The 20 faces

To find which corners make up each face, take the *convex hull*. Think of shrink-wrapping the 12 corners: the wrap
is made of 20 flat triangles, and those are the faces. `make_mesh.m` does this with `convhulln` (line 121); here
SciPy's `ConvexHull` does the same. Neighbouring corners are 63.4° apart as seen from the centre of the Earth
(line 122), which is an arc of about 7,062 km on the surface. This angle is called Ω below.

In [ ]:
T = ConvexHull(V).simplices          # 20 rows of 3 corner indices
Om = np.arccos(V[T[0, 0]] @ V[T[0, 1]])
print('faces:', len(T), ' edge angle: %.1f deg' % np.degrees(Om), ' edge arc: %.0f km' % (R * Om))

ax = plt.figure(figsize=(5, 5)).add_subplot(projection='3d')
for f in T:
    loop = V[np.r_[f, f[0]]]
    ax.plot(*loop.T, color='0.3', lw=0.8)
ax.scatter(*V.T, color='crimson', s=40)
ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()
ax.set_title('12 corners (red), 20 faces')
plt.show()

## 3. Fill one face with a triangle lattice

Pick a whole number n, the number of steps along each face edge. Every point on a face gets three counts
(i, j, k) with i + j + k = n, much like (i, j) on a MOM6 grid. If the face's corners are A, B and C, the point is

  sin(i·Ω/n)·A + sin(j·Ω/n)·B + sin(k·Ω/n)·C,

pushed back out onto the sphere by dividing by its length (lines 129–131 and 134–135).

Why sine and not plain i/n? Along an edge one count is 0, and the formula becomes "spherical linear interpolation",
which spaces the points exactly evenly along the curved edge. Two neighbouring faces then produce the same points on
their shared edge, so the faces fit together point for point.

In [ ]:
def face_points(A, B, C, n):
    # all lattice points of one face, n steps per edge (make_mesh.m lines 129-135)
    I, J = np.meshgrid(np.arange(n + 1), np.arange(n + 1), indexing='ij')
    m = I + J <= n
    W = np.sin(np.stack([I[m], J[m], n - I[m] - J[m]], axis=1) / n * Om)
    X = W @ np.array([A, B, C])
    return X / np.linalg.norm(X, axis=1, keepdims=True)

def lonlat(X):
    return np.degrees(np.arctan2(X[:, 1], X[:, 0])), np.degrees(np.arcsin(np.clip(X[:, 2], -1, 1)))

n = 6
X = face_points(*V[T[0]], n)
print('points on one face with n = %d: %d  (= (n+1)(n+2)/2 = %d)' % (n, len(X), (n + 1) * (n + 2) // 2))
lo, la = lonlat(X); clo, cla = lonlat(V[T[0]])
plt.figure(figsize=(5, 4))
plt.scatter(lo, la, s=15); plt.scatter(clo, cla, color='crimson', s=40, label='face corners')
plt.xlabel('lon'); plt.ylabel('lat'); plt.legend(); plt.title('one face, n = %d steps per edge' % n)
plt.show()

## 4. All 20 faces, without duplicates

Do the same for all 20 faces. The points on shared edges and corners now appear two or more times (an edge belongs
to 2 faces, a corner to 5), so remove the copies. `make_mesh.m` uses `uniquetol` (line 140); here a k-d tree finds
pairs closer than 1e-9. The result should be exactly 10n² + 2 points: 20 faces of n²/2 small triangles each give
10n² triangles, and a sphere covered by T triangles has T/2 + 2 points.

In [ ]:
def icosphere_points(n):
    X = np.concatenate([face_points(*V[f], n) for f in T])
    dup = np.zeros(len(X), bool)
    for a, b in cKDTree(X).query_pairs(1e-9):
        dup[max(a, b)] = True                    # keep one copy of each shared point
    return X[~dup]

for n in (1, 2, 6, 20):
    P = icosphere_points(n)
    print('n = %2d: %6d points, 10n^2+2 = %6d' % (n, len(P), 10 * n * n + 2))

## 5. Connect the dots: 6 neighbours everywhere, except 12 fives

Points on a sphere can be joined into triangles with the same convex-hull trick as in step 2. Here the hull's
triangles are exactly the lattice's small triangles; for n = 4 and n = 20 I checked that they give the same edges as
joining the lattice by hand. Then count each node's neighbours. On a MOM6 grid every ocean point has 4 neighbours,
implied by (i, j). Here the neighbours have to be listed, and in a perfect triangle lattice there are 6. The 12
icosahedron corners have only 5. That can't be avoided: for any triangle mesh covering a sphere, adding up (6 −
neighbours) over all nodes gives exactly 12 (Euler's formula). Those 12 five-neighbour nodes are the soccer ball's
pentagons.

In [ ]:
def edges_of(P):
    H = ConvexHull(P).simplices                                  # triangles
    E = np.sort(np.concatenate([H[:, [0, 1]], H[:, [1, 2]], H[:, [2, 0]]]), axis=1)
    return H, np.unique(E, axis=0)                               # each edge once

P = icosphere_points(20)
H, E = edges_of(P)
deg = np.bincount(E.ravel(), minlength=len(P))
print('neighbour counts:', dict(zip(*np.unique(deg, return_counts=True))))
print('sum of (6 - neighbours):', (6 - deg).sum())
lo, la = lonlat(P)
plt.figure(figsize=(8, 4))
plt.scatter(lo, la, s=2, color='0.6')
plt.scatter(lo[deg == 5], la[deg == 5], s=40, color='crimson', label='5 neighbours')
plt.legend(loc='lower left'); plt.xlabel('lon'); plt.ylabel('lat'); plt.title('n = 20: %d points' % len(P))
plt.show()

## 6. From a target spacing to n

We want the triangles to have sides of about h (25 km in the open ocean). In a lattice of equilateral triangles,
each point owns two triangles, an area of (√3/2)·h². Setting 10n² points × (√3/2)·h² equal to the sphere's area 4πR²
gives n = (R/h)·√(4π/(5√3)) (line 128; the comment on lines 126–127 says why this beats edge-arc/h). At 25 km this
gives n = 307, about 0.94 million points on the whole sphere, before removing land and before adding the finer coastal
zones. We only count the points here, not build them.

In [ ]:
def n_for_spacing(h_km):
    return max(1, round(R / h_km * np.sqrt(4 * np.pi / (5 * np.sqrt(3)))))   # make_mesh.m line 128

for h in (100, 50, 25, 12.5, 10, 5):
    n = n_for_spacing(h)
    print('h = %5.1f km: n = %5d, points on the whole sphere = %11s' % (h, n, format(10 * n * n + 2, ',')))

## 7. The small built-in stretch

The lattice isn't perfectly even. Edges lying along the 30 icosahedron edges are the shortest, at Ω/n, about 0.92h.
Edges in the middle of each face are the longest, up to about 1.08h. At h = 25 km that is roughly 23 km to 27 km, a
max/min ratio of about 1.18 (the comment on line 111 says ~17%). The ratio is the same for every n, so a modest n shows
the pattern. In the real mesh the same thing appears as ~23 km lines along the icosahedron edges and ~27 km face
centres, and DistMesh's relaxation hardly evens it out: after relaxation the open-ocean edge lengths still run from
23.3 km (5th percentile) to 26.8 km (95th), or 26.7 km with 700 iterations (`mesh/README.md`, Experiment section).

In [ ]:
n = 48
P = icosphere_points(n)
H, E = edges_of(P)
h = R * np.sqrt(4 * np.pi / (5 * np.sqrt(3))) / n               # the spacing this n stands for
L = R * np.arccos(np.clip((P[E[:, 0]] * P[E[:, 1]]).sum(1), -1, 1))
r = L / h
print('edge length / h: min %.3f, p5 %.3f, median %.3f, p95 %.3f, max %.3f, max/min %.3f'
      % (r.min(), *np.percentile(r, [5, 50, 95]), r.max(), r.max() / r.min()))
print('scaled to h = 25 km: p5 %.1f, median %.1f, p95 %.1f km' % tuple(25 * np.percentile(r, [5, 50, 95])))

# mean length of the edges at each node, as if h were 25 km
s = np.bincount(E[:, 0], r, len(P)) + np.bincount(E[:, 1], r, len(P))
c = np.bincount(E.ravel(), minlength=len(P))
lo, la = lonlat(P)
plt.figure(figsize=(9, 4.2))
sc = plt.scatter(lo, la, c=25 * s / c, s=4, cmap='viridis')
plt.colorbar(sc, label='mean edge length (km, at h = 25 km)')
plt.xlabel('lon'); plt.ylabel('lat'); plt.title('short edges trace the icosahedron edges; face middles are long')
plt.show()

## 8. Using it, part 1: keep only ocean points

WW3 only needs nodes in the water. In the real pipeline OceanMesh2D's distance-to-coast function drops points on land
(`make_mesh.m` line 49). Here a made-up "land" of two round caps stands in for the coastline. This is a toy mask, not
real geography.

In [ ]:
def cap(lon0, lat0, radius_deg):
    c = np.array([np.cos(np.radians(lat0)) * np.cos(np.radians(lon0)),
                  np.cos(np.radians(lat0)) * np.sin(np.radians(lon0)), np.sin(np.radians(lat0))])
    return c, radius_deg

LAND = [cap(20, 10, 35), cap(-100, 40, 25)]
def is_land(X):
    return np.any([np.degrees(np.arccos(np.clip(X @ c, -1, 1))) < rad for c, rad in LAND], axis=0)

P = icosphere_points(24)
ocean = P[~is_land(P)]
lo, la = lonlat(ocean)
plt.figure(figsize=(8, 4))
plt.scatter(lo, la, s=2)
plt.xlabel('lon'); plt.ylabel('lat'); plt.title('%d of %d points kept as ocean (toy land)' % (len(ocean), len(P)))
plt.show()

## 9. Using it, part 2: several sizes at once

The real mesh isn't 25 km everywhere: it is finer near coasts, in tropical-cyclone regions and along US coasts. The
pipeline handles this in three steps:
1. List a set of sizes ("levels"), spaced by at most 10% (`size_levels`, lines 64–74).
2. Build one complete soccer-ball lattice per level.
3. Keep each point only where its level is the nearest to the size wanted there (lines 125–142, the comparison being
   on line 137).

Far from any size change, the result is one clean lattice: only its 12 corners have 5 neighbours. Where the size
changes, lattices of different n meet along seams, and the seams are where nodes with other than 6 neighbours collect.
`mesh/README.md` says the same of the real mesh: the defects left are at the edges of the 25 km zone, because changing
size needs them. Below, a toy size is 150 km along a circle (standing in for a coastline; both sides are kept here) and grows to
300 km away from it. The seam triangles
here come straight from the hull, before any relaxation, so they are messier than in the real mesh.

In [ ]:
c0, rad0 = cap(0, 20, 30)
def dist_coast_km(X):                              # distance from the toy 'coastline' circle
    return np.abs(np.degrees(np.arccos(np.clip(X @ c0, -1, 1))) - rad0) * np.pi / 180 * R
hmin, hmax = 150.0, 300.0
def size_wanted(X):                                # like h = min(25, 10 + 0.15 d) in the real mesh
    return np.minimum(hmax, hmin + 0.15 * dist_coast_km(X))

nlev = int(np.ceil(np.log(hmax / hmin) / np.log(1.1)))
lev = hmin * (hmax / hmin) ** (np.arange(nlev + 1) / nlev)       # <= 10% steps
pts, lab = [], []
for k, hk in enumerate(lev):
    X = icosphere_points(n_for_spacing(hk))
    nearest = np.argmin(np.abs(np.log(size_wanted(X))[:, None] - np.log(lev)[None, :]), axis=1)
    pts.append(X[nearest == k]); lab.append(np.full((nearest == k).sum(), k))
P, lab = np.concatenate(pts), np.concatenate(lab)
H, E = edges_of(P)
deg = np.bincount(E.ravel(), minlength=len(P))
far = dist_coast_km(P) > (hmax - hmin) / 0.15 + 500       # well inside the coarsest zone
print('levels (km):', np.round(lev, 1))
print('%d points; nodes without 6 neighbours: %d in all, %d far from the size change (the icosahedron corners)'
      % (len(P), (deg != 6).sum(), (deg[far] != 6).sum()))
lo, la = lonlat(P)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sc = ax[0].scatter(lo, la, c=lev[lab], s=3, cmap='viridis'); fig.colorbar(sc, ax=ax[0], label='level size (km)')
ax[0].set_title('which lattice each point came from')
ax[1].scatter(lo, la, s=1, color='0.75')
ax[1].scatter(lo[deg != 6], la[deg != 6], s=6, color='crimson')
ax[1].set_title('nodes without 6 neighbours (red)')
for a in ax: a.set_xlabel('lon'); a.set_ylabel('lat')
plt.show()

## 10. The shaking, in words

After step 9, OceanMesh2D's mesh generator (DistMesh) takes over (`make_mesh.m` lines 42–51). Picture every edge as a
spring that wants to be the target size at its location. Each iteration moves the points a little in the direction
the springs pull, rebuilds the triangles, and every so often removes points that are far too close and adds points
where edges are far too long. It stops after `itmax` iterations (default 100). That is a simplified picture; the
real code is OceanMesh2D's `meshgen.m`.

What the shaking can and can't do (from `mesh/README.md`):
- It evens out the spacing locally, but it can't remove the pattern of 5- and 7-neighbour nodes; that is fixed mostly
  by the starting points. Compared with Alper's latitude-row start (v2), this icosahedral start cut open-ocean
  5/7-neighbour nodes from 5,041 to 1,464, and those deep in the open ocean from 2,693 to 2.
- It doesn't remove the stretch from step 7. With 700 iterations instead of 100, the open-ocean edge lengths still
  ran 23.3–26.7 km (against 23.3–26.8 km). Shaking spreads changes out like diffusion, roughly N² iterations for a
  pattern N spacings wide, and an icosahedron face is about 280 spacings across. The fix to try next is an
  equal-area starting map.

## The real pipeline, in short

```bash
cd ~/documents/mesawave/mesh/mesawave_mesh
# mesh with the icosahedral start: PREFIX SCALE SIZEFILE icos [ITMAX]; SCALE 4 is a quick ~80k-node test
qcmd -A NCGD0011 -q develop -l select=1:ncpus=16:mem=200GB -l walltime=06:00:00 -- \
  $PWD/run_mesh.sh try1 4 sizefx_na1000_ot500_hull.nc icos
# then: checks, WW3 grid inputs, and (for CESM) tools/stage_grid.sh; see mesh/README.md
```